Filter only pdf/announcements that have the most traffic to narrow down the number of articles to process


Process into weekly sentiment scores

# Data Acquisition Process

In [1]:
# CNINFO CSI300 announcement metadata scraper (MVP)
# Goal: write announcements_meta_2025-12.csv with: ticker, publish_ts(UTC), title, pdf_url, source, column, orgId

import csv, re, time, random, requests
from datetime import datetime, timezone

# ---- Config ----
CSI300_CONSTITUENTS_URL = "https://yfiua.github.io/index-constituents/2026/01/constituents-csi300.csv"
SZSE_STOCK_JSON = "https://www.cninfo.com.cn/new/data/szse_stock.json"  # contains orgId mapping (incl. SH codes in your run)
CNINFO_QUERY_URL = "https://www.cninfo.com.cn/new/hisAnnouncement/query"
CNINFO_STATIC_BASE = "https://static.cninfo.com.cn/"
DATE_RANGE = "2025-12-01~2025-12-31"
OUT_CSV = "announcements_meta_2025-12.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://www.cninfo.com.cn",
    "Referer": "https://www.cninfo.com.cn/",
}


In [2]:
# ---- Helpers ----
def extract_6digit_code(sym: str) -> str | None:
    m = re.search(r"(\d{6})", sym or "")
    return m.group(1) if m else None

def get_csi300_codes(url: str) -> list[str]:
    r = requests.get(url, headers={"User-Agent": HEADERS["User-Agent"]}, timeout=30)
    r.raise_for_status()
    rows = list(csv.DictReader(r.text.splitlines()))
    # yfiua uses Yahoo-style symbols like 000001.SZ
    col = "Symbol" if "Symbol" in rows[0] else list(rows[0].keys())[0]
    codes = [extract_6digit_code(row.get(col, "")) for row in rows]
    codes = sorted({c for c in codes if c})
    return codes

def load_org_map_szse(url: str) -> dict[str, str]:
    r = requests.get(url, headers={"User-Agent": HEADERS["User-Agent"]}, timeout=30)
    r.raise_for_status()
    data = r.json()
    return {it["code"]: it["orgId"] for it in data.get("stockList", []) if it.get("code") and it.get("orgId")}

def infer_column(code: str) -> str:
    return "sse" if code.startswith("6") else "szse"

def cninfo_query(code: str, orgid: str, page: int, page_size: int = 30) -> dict:
    payload = {
        "pageNum": page,
        "pageSize": page_size,
        "tabName": "fulltext",
        "column": infer_column(code),
        "stock": f"{code},{orgid}",
        "seDate": DATE_RANGE,
        "sortName": "time",
        "sortType": "desc",
        "isHLtitle": "true",
        # keep optional filters blank for MVP
        "searchkey": "", "secid": "", "plate": "", "trade": "", "category": "",
    }
    r = requests.post(CNINFO_QUERY_URL, headers=HEADERS, data=payload, timeout=30)
    r.raise_for_status()
    return r.json()

def to_utc_string_from_ms(ms: int) -> str:
    # timezone-aware UTC (avoids utcfromtimestamp deprecation)
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S+00:00")

def scrape_one_code(code: str, orgid: str, max_pages: int = 50) -> list[dict]:
    rows = []
    for page in range(1, max_pages + 1):
        data = cninfo_query(code, orgid, page=page, page_size=30)
        anns = data.get("announcements") or []
        if not anns:
            break

        for a in anns:
            pub_str = a.get("announcementTimeStr")
            pub_ms = a.get("announcementTime")
            publish_ts = pub_str or (to_utc_string_from_ms(pub_ms) if pub_ms else "")
            adjunct = a.get("adjunctUrl", "")
            pdf_url = (CNINFO_STATIC_BASE + adjunct) if adjunct else ""

            rows.append({
                "ticker": code,
                "publish_ts": publish_ts,
                "title": a.get("announcementTitle", ""),
                "pdf_url": pdf_url,
                "source": "CNINFO",
                "column": infer_column(code),
                "orgId": orgid,
            })

        if data.get("hasMore") is False:
            break

        time.sleep(random.uniform(0.7, 1.6))  # polite per-page throttling
    return rows


In [3]:
# ---- Main ----
codes = get_csi300_codes(CSI300_CONSTITUENTS_URL)
org_map_full = load_org_map_szse(SZSE_STOCK_JSON)

# In your run, szse_stock.json covered all CSI300 codes (including 6xxxxx)
org_map = {c: org_map_full[c] for c in codes if c in org_map_full}

print(f"CSI300 codes: {len(codes)} | with orgId: {len(org_map)}")

all_rows = []
codes_with_org = sorted(org_map.keys())

for i, code in enumerate(codes_with_org, 1):
    try:
        all_rows.extend(scrape_one_code(code, org_map[code]))
        if i % 10 == 0:
            print(f"[{i}/{len(codes_with_org)}] cumulative rows: {len(all_rows)}")
    except Exception as e:
        print(f"[{i}/{len(codes_with_org)}] {code}: ERROR {e}")
    time.sleep(random.uniform(0.4, 1.1))  # polite per-ticker throttling

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["ticker","publish_ts","title","pdf_url","source","column","orgId"])
    w.writeheader()
    w.writerows(all_rows)

print(f"Wrote {len(all_rows)} rows to {OUT_CSV}")

CSI300 codes: 300 | with orgId: 300
[10/300] cumulative rows: 191
[20/300] cumulative rows: 319
[30/300] cumulative rows: 448
[40/300] cumulative rows: 652
[50/300] cumulative rows: 802
[60/300] cumulative rows: 941
[70/300] cumulative rows: 1227
[80/300] cumulative rows: 1352
[90/300] cumulative rows: 1463
[100/300] cumulative rows: 1555
[110/300] cumulative rows: 1656
[120/300] cumulative rows: 1741
[130/300] cumulative rows: 1946
[140/300] cumulative rows: 2056
[150/300] cumulative rows: 2160
[160/300] cumulative rows: 2338
[170/300] cumulative rows: 2436
[180/300] cumulative rows: 2531
[190/300] cumulative rows: 2607
[200/300] cumulative rows: 2787
[210/300] cumulative rows: 2975
[220/300] cumulative rows: 3072
[230/300] cumulative rows: 3163
[240/300] cumulative rows: 3317
[250/300] cumulative rows: 3398
[260/300] cumulative rows: 3516
[270/300] cumulative rows: 3643
[280/300] cumulative rows: 3762
[290/300] cumulative rows: 3881
[300/300] cumulative rows: 4031
Wrote 4031 rows to 

In [4]:
# Quick validation (sanity checks)
import pandas as pd

df = pd.read_csv("announcements_meta_2026-01.csv")
print("Rows:", len(df), "| Unique tickers:", df["ticker"].nunique())
print(df.head(3))

print("Empty pdf_url %:", (df["pdf_url"].fillna("") == "").mean())
ts = pd.to_datetime(df["publish_ts"], errors="coerce", utc=True)
print("Bad timestamps %:", ts.isna().mean())
print("Shanghai row share:", df["ticker"].astype(str).str.startswith("6").mean())
print("Duplicate pdf_url %:", df["pdf_url"].duplicated().mean())

Rows: 1988 | Unique tickers: 287
   ticker                 publish_ts        title  \
0       1  2026-01-26 16:00:00+00:00  关于选举职工董事的公告   
1       1  2026-01-23 16:00:00+00:00      董事会决议公告   
2       1  2026-01-23 16:00:00+00:00       关联交易公告   

                                             pdf_url  source column  \
0  https://static.cninfo.com.cn/finalpage/2026-01...  CNINFO   szse   
1  https://static.cninfo.com.cn/finalpage/2026-01...  CNINFO   szse   
2  https://static.cninfo.com.cn/finalpage/2026-01...  CNINFO   szse   

         orgId  
0  gssz0000001  
1  gssz0000001  
2  gssz0000001  
Empty pdf_url %: 0.0
Bad timestamps %: 0.0
Shanghai row share: 0.6217303822937625
Duplicate pdf_url %: 0.0


# PDF download with caching

In [6]:
# Download PDFs with caching
from pathlib import Path
import os, time, random, hashlib
import pandas as pd
import requests
from urllib.parse import urlparse


Sample 5 pdf URLs:
['https://static.cninfo.com.cn/finalpage/2025-12-20/1224888992.PDF', 'https://static.cninfo.com.cn/finalpage/2025-12-17/1224882117.PDF', 'https://static.cninfo.com.cn/finalpage/2025-12-17/1224882115.PDF', 'https://static.cninfo.com.cn/finalpage/2025-12-17/1224882116.PDF', 'https://static.cninfo.com.cn/finalpage/2025-12-13/1224876133.PDF']
Random sample titles:
['洛阳钼业债券持有人会议规则', '恒瑞医药第九届董事会第二十次会议决议公告', '中海油田服务股份有限公司章程（2025年修订）', '第六届董事会第二十六次会议决议公告', '三一重工股份有限公司关于取消公司监事会并修订《公司章程》《股东会议事规则》《董事会议事规则》的公告']


In [5]:
# --- Config ---
META_GLOB = "announcements_meta_2025-12.csv"
PDF_DIR = Path("cninfo_pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT = 45
SLEEP_BETWEEN = (0.2, 0.8)  # polite jitter; adjust if you're getting throttled
USER_AGENT = "Mozilla/5.0 (compatible; CapstonePDFDownloader/1.0; +https://example.com)"

def pick_latest_meta(glob_pat=META_GLOB) -> Path:
    cands = sorted(Path(".").glob(glob_pat))
    if not cands:
        raise FileNotFoundError(f"No metadata file found matching {glob_pat} in {Path('.').resolve()}")
    return max(cands, key=lambda p: p.stat().st_mtime)

def safe_filename(s: str, max_len: int = 120) -> str:
    s = (s or "").strip()
    s = "".join(ch if ch.isalnum() or ch in ("-", "_", ".", " ") else "_" for ch in s)
    s = " ".join(s.split())
    return (s[:max_len] if len(s) > max_len else s) or "untitled"

def sha1_bytes(b: bytes) -> str:
    h = hashlib.sha1()
    h.update(b)
    return h.hexdigest()

def download_one(url: str, out_path: Path) -> dict:
    """
    Download URL to out_path if not already present.
    Returns dict with status info.
    """
    if out_path.exists() and out_path.stat().st_size > 0:
        return {"status": "cached", "bytes": out_path.stat().st_size}

    headers = {"User-Agent": USER_AGENT}
    try:
        r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        data = r.content
        if len(data) < 1024:  # tiny file likely indicates an HTML error page
            return {"status": "too_small", "bytes": len(data), "note": "Possible HTML/error page returned"}
        out_path.write_bytes(data)
        return {"status": "downloaded", "bytes": len(data), "sha1": sha1_bytes(data)}
    except Exception as e:
        return {"status": "error", "error": str(e)}


In [ ]:
# --- Load metadata ---
meta_path = pick_latest_meta()
print("Using metadata:", meta_path)

df_meta = pd.read_csv(meta_path)
required_cols = {"ticker", "title", "pdf_url"}
missing = required_cols - set(df_meta.columns)
assert not missing, f"Metadata missing required columns: {missing}"

# Optional columns if present: publish_ts, ann_id, exchange/source
# We'll build a deterministic file name even if ann_id isn't available.
def build_pdf_path(row) -> Path:
    ticker = str(row.get("ticker", "")).zfill(6)
    title = safe_filename(str(row.get("title", "")))
    # try to include date if present
    date_str = ""
    if "publish_ts" in row and pd.notnull(row["publish_ts"]):
        try:
            # publish_ts might be ISO or epoch; handle both
            ts = row["publish_ts"]
            if isinstance(ts, (int, float)) or (isinstance(ts, str) and ts.isdigit()):
                dt = pd.to_datetime(int(ts), unit="s", utc=True).tz_convert("Asia/Shanghai")
            else:
                dt = pd.to_datetime(ts, utc=True).tz_convert("Asia/Shanghai")
            date_str = dt.strftime("%Y%m%d")
        except Exception:
            date_str = ""
    stem = "_".join([x for x in [ticker, date_str, title] if x])
    return PDF_DIR / f"{stem}.pdf"


In [ ]:
# --- Download loop ---
results = []
for _, row in df_meta.iterrows():
    url = row["pdf_url"]
    out_path = build_pdf_path(row)
    res = download_one(url, out_path)
    results.append({
        "pdf_path": str(out_path),
        "download_status": res.get("status"),
        "download_bytes": res.get("bytes"),
        "download_sha1": res.get("sha1"),
        "download_error": res.get("error"),
    })
    time.sleep(random.uniform(*SLEEP_BETWEEN))

df_dl = pd.concat([df_meta.reset_index(drop=True), pd.DataFrame(results)], axis=1)

print(df_dl["download_status"].value_counts(dropna=False))
print("Example saved file:", df_dl.loc[df_dl["download_status"].isin(["downloaded","cached"])].head(1)["pdf_path"].tolist())

# Persist for the next stages
df_dl.to_csv("announcements_with_pdf_paths.csv", index=False)
print("Wrote announcements_with_pdf_paths.csv")


# PDF text extraction


In [7]:
# Install if needed:
# !pip -q install pymupdf pdfplumber pytesseract pillow pyarrow tqdm
!pip -q install pymupdf pdfplumber pytesseract pillow pyarrow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 25.7 MB/s eta 0:00:00


In [ ]:
# PDF text extraction (CNINFO announcements)
# Strategy:
#   1) Try native text extraction (PyMuPDF, then pdfplumber)
#   2) If too little text, optionally OCR fallback (requires Tesseract + chi_sim)
# Output:
#   announcements_text.parquet / .csv with extracted text, method, diagnostics

from pathlib import Path
import re, hashlib
import pandas as pd

from tqdm.auto import tqdm

import fitz  # PyMuPDF
import pdfplumber

OCR_ENABLED = True
try:
    import pytesseract
    from PIL import Image
except Exception:
    OCR_ENABLED = False
    print("OCR disabled (pytesseract/PIL not available). Native extraction only.")

IN_PATH = Path("announcements_with_pdf_paths.csv")
assert IN_PATH.exists(), "Run the PDF download section first to generate announcements_with_pdf_paths.csv"
df = pd.read_csv(IN_PATH)

def normalize_cn_text(s: str) -> str:
    if not isinstance(s, str) or not s:
        return ""
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    s = re.sub(r"(?m)^\s*第\s*\d+\s*页\s*/\s*共\s*\d+\s*页\s*$", "", s)
    return s.strip()

def sha1_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def extract_text_pymupdf(path: Path, max_pages=None) -> str:
    parts = []
    with fitz.open(path) as doc:
        end = min(doc.page_count, max_pages) if max_pages else doc.page_count
        for i in range(end):
            parts.append(doc.load_page(i).get_text("text") or "")
    return "\n".join(parts)

def extract_text_pdfplumber(path: Path, max_pages=None) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        end = min(len(pdf.pages), max_pages) if max_pages else len(pdf.pages)
        for i in range(end):
            parts.append(pdf.pages[i].extract_text() or "")
    return "\n".join(parts)

def ocr_pdf(path: Path, dpi: int = 220, max_pages=None) -> str:
    if not OCR_ENABLED:
        return ""
    parts = []
    with fitz.open(path) as doc:
        end = min(doc.page_count, max_pages) if max_pages else doc.page_count
        for i in range(end):
            pix = doc.load_page(i).get_pixmap(dpi=dpi)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            parts.append(pytesseract.image_to_string(img, lang="chi_sim"))
    return "\n".join(parts)

def extract_cninfo_pdf_text(path: Path, min_chars: int = 200) -> dict:
    meta = {"method": "failed", "char_count": 0, "error": None, "page_count": None, "text": ""}
    try:
        with fitz.open(path) as doc:
            meta["page_count"] = doc.page_count
    except Exception:
        pass

    # 1) PyMuPDF
    try:
        t = normalize_cn_text(extract_text_pymupdf(path))
        if len(t) >= min_chars:
            meta.update(method="pymupdf", char_count=len(t), text=t)
            return meta
    except Exception as e:
        meta["error"] = f"pymupdf_error: {e}"

    # 2) pdfplumber
    try:
        t = normalize_cn_text(extract_text_pdfplumber(path))
        if len(t) >= min_chars:
            meta.update(method="pdfplumber", char_count=len(t), text=t)
            return meta
    except Exception as e:
        meta["error"] = (meta["error"] or "") + f" | pdfplumber_error: {e}"

    # 3) OCR
    if OCR_ENABLED:
        try:
            t = normalize_cn_text(ocr_pdf(path))
            if len(t) >= min_chars:
                meta.update(method="ocr", char_count=len(t), text=t)
                return meta
        except Exception as e:
            meta["error"] = (meta["error"] or "") + f" | ocr_error: {e}"

    # fall back to whatever we got (could be short)
    meta["text"] = normalize_cn_text(meta["text"])
    meta["char_count"] = len(meta["text"])
    return meta

# Run extraction
records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting PDF text"):
    pdf_path = Path(row["pdf_path"])
    if not pdf_path.exists() or pdf_path.stat().st_size == 0:
        records.append({"text": "", "text_method": "missing", "char_count": 0, "page_count": None, "text_error": "missing_file", "file_sha1": None})
        continue
    out = extract_cninfo_pdf_text(pdf_path)
    records.append({
        "text": out["text"],
        "text_method": out["method"],
        "char_count": out["char_count"],
        "page_count": out["page_count"],
        "text_error": out["error"],
        "file_sha1": sha1_file(pdf_path),
    })

df_text = pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)

print(df_text["text_method"].value_counts(dropna=False))
print(df_text["char_count"].describe(percentiles=[.1,.25,.5,.75,.9]))

# Persist
df_text.to_parquet("announcements_text.parquet", index=False)
df_text.to_csv("announcements_text.csv", index=False, encoding="utf-8-sig")
print("Wrote announcements_text.parquet and announcements_text.csv")


# Doc sentiment scoring (lexicon baseline)


# Bar aggregation to 15-min in Asia/Shanghai